## Key steps:
### 1. Convert a user's question into a TRAPI json using a query template (LLM facilitated)
### 2. Validate the TRAPI json format
### 3. Refine the TRAPI json format by selecting the similar categories and predicates (LLM facilitated) and user's selection
### 4. ID formatting
### 5. Query, rank, and visualization

![image.png](../Figures/LLM_TRAPI_converting2.png)

In [ ]:
from TCT import TCT, translator_query
from TCT import format_id, get_similar_category, get_similar_predicate
from TCT.translator_resources import TranslatorResources
import pandas as pd
import json
import ipywidgets as widgets
from IPython.display import display

# Load Translator resources
resources = TranslatorResources.load()

# Derive predicates and categories from the metaKG
All_predicates = list(resources.meta_kg['Predicate'].unique())
KG_category = list(set(list(resources.meta_kg['Subject'].unique()) + list(resources.meta_kg['Object'].unique())))
KG_predicates = list(resources.meta_kg['Predicate'].unique())

# Build triplets dict using 'Predicate' column
metaKG_triplets = resources.meta_kg[['Subject', 'Predicate', 'Object']]
metaKG_triplets_dic = metaKG_triplets.to_dict('records')

# Load the query template
query_json_temp = TCT.load_json_template()
query_json = str(query_json_temp)
print(query_json)

In [ ]:
import openai
import os

# Configure OpenAI API key from environment variable
openai.api_key = os.environ.get('OPENAI_API_KEY', '')
if not openai.api_key:
    print("Warning: OPENAI_API_KEY not set. Set it via: export OPENAI_API_KEY='your-key'")

In [ ]:
def convert_Question2Query(question):
    """Convert a natural language question to a TRAPI query JSON using ChatGPT."""
    input_text = (
        "We know the available predicates in the KG are: " + ','.join(list(set(KG_predicates)))
        + ". We also know the available categories in the KGs are " + ','.join(list(set(KG_category)))
        + ". We also know a TRAPI message template is " + query_json
        + ". With the question of " + question
        + " What is the json format of message to represent this question? "
        + "The following rules for the output: "
        + "1) The result must be just a json format with the same structure with template; "
        + "2) categories should be replaced from the categories in the KG; "
        + "3) predicates can be replaced from the predicates in the KG; "
        + "4) can use the name to fill the ids; "
        + "4) the output must start with '{' and end with '}', and be a standard json format. "
        + "At least one ids should be given and No annotations are needed!"
    )
    query_json_cur = TCT.ask_chatGPT4(input_text)
    query_json_cur_clean = TCT.extract_json(query_json_cur)
    return query_json_cur_clean

In [ ]:
def reverse_nodes(query_json_input):
    """Swap subject (n0) and object (n1) nodes."""
    original_subject = query_json_input['message']['query_graph']['nodes']['n0']
    original_object = query_json_input['message']['query_graph']['nodes']['n1']
    query_json_input['message']['query_graph']['nodes']['n0'] = original_object
    query_json_input['message']['query_graph']['nodes']['n1'] = original_subject
    return query_json_input


def check_json(query_json_input):
    """Validate that the query JSON contains valid categories and predicates from the metaKG."""
    Label = False
    subject_category = []
    object_category = []
    predicate = []
    triple_pass = 0

    all_categories = set(list(resources.meta_kg['Subject']) + list(resources.meta_kg['Object']))
    all_predicates = set(resources.meta_kg['Predicate'])

    if 'message' in query_json_input:
        if 'query_graph' in query_json_input['message']:
            if 'nodes' in query_json_input['message']['query_graph']:
                if 'n0' in query_json_input['message']['query_graph']['nodes']:
                    if 'categories' in query_json_input['message']['query_graph']['nodes']['n0']:
                        overlap = set(query_json_input['message']['query_graph']['nodes']['n0']['categories']).intersection(all_categories)
                        if overlap:
                            subject_category = list(overlap)

                if 'n1' in query_json_input['message']['query_graph']['nodes']:
                    if 'categories' in query_json_input['message']['query_graph']['nodes']['n1']:
                        overlap = set(query_json_input['message']['query_graph']['nodes']['n1']['categories']).intersection(all_categories)
                        if overlap:
                            object_category = list(overlap)

            if 'edges' in query_json_input['message']['query_graph']:
                if 'e1' in query_json_input['message']['query_graph']['edges']:
                    if 'predicates' in query_json_input['message']['query_graph']['edges']['e1']:
                        overlap = set(query_json_input['message']['query_graph']['edges']['e1']['predicates']).intersection(all_predicates)
                        if overlap:
                            predicate = list(overlap)

    if subject_category and object_category and predicate:
        for subject_cur_category in subject_category:
            for object_cur_category in object_category:
                for predicate_cur in predicate:
                    if {'Subject': subject_cur_category, 'Predicate': predicate_cur, 'Object': object_cur_category} in metaKG_triplets_dic:
                        triple_pass += 1
                    if {'Subject': object_cur_category, 'Predicate': predicate_cur, 'Object': subject_cur_category} in metaKG_triplets_dic:
                        triple_pass += 1
                        query_json_input = reverse_nodes(query_json_input)
                        print("The reverse direction of the edge is correct! Subject and Object are reversed!")

    if triple_pass > 0:
        Label = True

    if Label:
        print("The json format is correct!")
    else:
        print("!!Incorrect input file format")
    return query_json_input

# Ask a question

In [ ]:
# Set the question
#question = "What genes or proteins interact with KRAS?"
#question = "What drugs may treat Type 2 diabetes?"
#question = "What are the drugs or small molecules that target KRAS?"
#question = "What diseases co-occurence with covid-19?"
question = "What drug increase the risk of liver cancer?"
#question = "What drugs may treat acute myeloid leukemia?"
#question = "What symptoms are associated with long covid?"

In [ ]:
# Convert question to TRAPI query
query_json_cur_clean = convert_Question2Query(question)
query_json_cur_clean

In [ ]:
# Validate the query
query_json_cur_clean = check_json(query_json_cur_clean)

In [ ]:
# Optional: switch subject and object
switch_subject_object = widgets.RadioButtons(
    options=['Yes', 'No'],
    value='No',
    description='Switch subject and object?',
    disabled=False)
display(switch_subject_object)

In [ ]:
if switch_subject_object.value == 'Yes':
    query_json_cur_clean = reverse_nodes(query_json_cur_clean)
print(query_json_cur_clean)

In [ ]:
# Optional: refine categories
refine_category = widgets.RadioButtons(
    options=['Yes', 'No'],
    value='No',
    description='Refine category?',
    disabled=False)
display(refine_category)

In [ ]:
if refine_category.value == 'Yes':
    similar_category = get_similar_category(query_json_cur_clean, KG_category)
    print(query_json_cur_clean)

    category_n1 = widgets.SelectMultiple(
        options=similar_category, value=[], description='Node 0', disabled=False,
        layout=widgets.Layout(width='80%', height='300px'))
    display(category_n1)

    category_n2 = widgets.SelectMultiple(
        options=similar_category, value=[], description='Node 1', disabled=False,
        layout=widgets.Layout(width='80%', height='300px'))
    display(category_n2)

In [ ]:
# Update categories if refined
if refine_category.value == 'Yes':
    if len(category_n1.value) > 0:
        print("updated node 1!")
        query_json_cur_clean['message']['query_graph']['nodes']['n0']['categories'] = list(category_n1.value)
    if len(category_n2.value) > 0:
        print("updated node 2!")
        query_json_cur_clean['message']['query_graph']['nodes']['n1']['categories'] = list(category_n2.value)

# Optional: refine predicates
refine_predicates = widgets.RadioButtons(
    options=['Yes', 'No'],
    value='No',
    description='Refine predicates?',
    disabled=False)
display(refine_predicates)

In [ ]:
if refine_predicates.value == 'Yes':
    print(question)
    print(query_json_cur_clean)
    similar_predicate = get_similar_predicate(query_json_cur_clean, All_predicates)

    predicate_e01 = widgets.SelectMultiple(
        options=similar_predicate, value=[], description='Predicates', disabled=False,
        layout=widgets.Layout(width='80%', height='300px'))
    display(predicate_e01)

# Update predicates if refined
if refine_predicates.value == 'Yes' and len(predicate_e01.value) > 0:
    query_json_cur_clean['message']['query_graph']['edges']['e1']['predicates'] = list(predicate_e01.value)

# Final validation
print("The current json format is:")
print(query_json_cur_clean)
TCT.TRAPI_json_validation(query_json_cur_clean, All_predicates, KG_category)

In [ ]:
# ID formatting: resolve names to CURIEs
query_json_cur_clean = format_id(query_json_cur_clean)
print(query_json_cur_clean)

In [ ]:
# Select APIs based on categories
input_node1_category = query_json_cur_clean['message']['query_graph']['nodes']['n0']['categories']
input_node2_category = query_json_cur_clean['message']['query_graph']['nodes']['n1']['categories']

sele_APIs = TCT.select_API(
    sub_list=input_node1_category,
    obj_list=input_node2_category,
    metaKG=resources.meta_kg)

print(f"Selected {len(sele_APIs)} APIs")
print(sele_APIs)

In [ ]:
# Rebuild query via format_query_json to fix n0/e1 -> n00/e00 key mismatch,
# then query APIs and parse results
subject_ids = query_json_cur_clean['message']['query_graph']['nodes']['n0'].get('ids', [])
object_ids = query_json_cur_clean['message']['query_graph']['nodes']['n1'].get('ids', [])
subject_categories = query_json_cur_clean['message']['query_graph']['nodes']['n0']['categories']
object_categories = query_json_cur_clean['message']['query_graph']['nodes']['n1']['categories']
predicates = query_json_cur_clean['message']['query_graph']['edges']['e1']['predicates']

query_json_rebuilt = TCT.format_query_json(
    subject_ids=subject_ids,
    object_ids=object_ids,
    subject_categories=subject_categories,
    object_categories=object_categories,
    predicates=predicates)

print("Rebuilt query:")
print(query_json_rebuilt)

# Query APIs
result = translator_query.parallel_api_query(
    query_json=query_json_rebuilt,
    select_APIs=list(sele_APIs),
    resources=resources,
    max_workers=len(sele_APIs))

# Parse results
result_parsed = TCT.parse_KG(result=result)

In [ ]:
# Rank results
if subject_ids:
    input_node1_id = subject_ids[0]
elif object_ids:
    input_node1_id = object_ids[0]
else:
    input_node1_id = None

result_ranked_by_primary_infores = TCT.rank_by_primary_infores(
    result_parsed=result_parsed,
    input_node=input_node1_id)

print(result_ranked_by_primary_infores.shape)
result_ranked_by_primary_infores.head(5)

In [ ]:
# Visualize results
TCT.visulization_one_hop_ranking(
    result_ranked_by_primary_infores=result_ranked_by_primary_infores,
    result_parsed=result_parsed,
    num_of_nodes=30,
    input_query=input_node1_id,
    fontsize=8)